# Magic Card To Text Dataset Generator  
__Objective:__ The aim of this notebook is to harness the IBM Granite Docling model to convert a large batch of Magic the Gathering card images into a structured text dataset. That dataset can then be used to fine tune a transformer model on multi-label classification of scryfall tags.

In [ ]:
# config
MODEL_NAME = "ibm-granite/granite-docling-258M"
MAX_SEQUENCE_LENGTH = 256

# CARD_EXTRACTION_PROMPT = 'Read every piece of visible text from this card.'
# CARD_EXTRACTION_PROMPT = 'Transcribe this Magic card exactly.'
CARD_EXTRACTION_PROMPT = '''
Transcribe this Magic card as plain text.
Do not include bounding boxes, coordinates, locations, XML tags, or markup.
'''

## Packages and Data

In [ ]:
# packages

## link directory
from pathlib import Path
import sys

workspace_root = Path.cwd()
if workspace_root.name == 'notebooks':
    workspace_root = workspace_root.parent
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

## custom packages
from src.card_ocr.dataset import load_manifest_records, summarize_manifest

In [4]:
# retrieve core data
manifest_path = workspace_root / "data" / "card_image_text_manifest.jsonl"
records = load_manifest_records(manifest_path, skip_missing_images=True)
summary = summarize_manifest(records)
summary


{'num_records': 8443, 'split_counts': {'train': 6713, 'val': 1680, 'test': 50}}

In [6]:
records[0]

{'id': 8079,
 'oracle_id': '4317f4f1-d339-4940-b46e-659641035595',
 'card_name': 'Ascended Lawmage',
 'split': 'train',
 'image_path': '/Users/nickcruickshank/Projects/scryfall-llm-sandbox/data/card_images/4317f4f1-d339-4940-b46e-659641035595.png',
 'image_type': 'png',
 'image_exists': True,
 'target_text': "Ascended Lawmage\n        Mana Cost = {2}{W}{U}\nMana Value = 4.0\n\n        Type Line = Creature — Vedalken Wizard\n\n        Rules Text = Flying\nHexproof (This creature can't be the target of spells or abilities your opponents control.)\n\n        Power = 3\nToughness = 2\n\n\n        Color Identity = ['U', 'W']\n\n        Rarity = uncommon",
 'tags': ['evasion', 'french vanilla']}

## Prepare Image Dataset

In [8]:
# import torch
from torch.utils.data import Dataset
from PIL import Image

class GraniteCardOCRDataset(Dataset):
    """
    Description
    ----------
    This class contains the dataset for the Granite OCR model which we will use 
    to extract the text from Magic the Gathering cards.

    Inputs
    ----------
    records = A list of dicts containing our dataset elements
    resize_height = The height to resize the images to. Defaultt to 384 pixels.
        Can likely acchieve higher quality resolution with up to 512 pixels,
        but would cost more compute.
    """
    def __init__(self, records, resize_height:int = 384):
        self.records = records
        self.resize_height = resize_height

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        # get the record 
        record = self.records[index]

        # retrieve the image and preprocess it
        with Image.open(record['image_path']) as image:
            # process the image
            image = image.convert('RGB')
            image = image.resize(
                (
                    int(image.width * self.resize_height / image.height),
                    self.resize_height
                )
            )

            out = {
                'id': record['oracle_id'],
                'card_name': record['card_name'],
                'image': image
            }

            return out

In [10]:
def get_tokenizer(processor):
    if hasattr(processor, 'tokenizer'):
        return processor.tokenizer
    raise AttributeError('Expected the Granite processor to expose a tokenizer')

In [15]:
def apply_chat_template(
    processor,
    messages, 
    add_generation_prompt:bool = False 
):    
    return processor.apply_chat_template(
        messages,
        add_generation_prompt = add_generation_prompt,
        tokenize = False
    )

def build_messages(
    prompt:str,
    target_text = None
):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt}
            ]
        }
    ]

    if target_text is not None:
        messages.append(
            {
                'role': 'assistant',
                'content': [{'type': 'text', 'text': target_text}]
            }
        )

    return messages

In [ ]:
class GraniteCardOCRCollator:
    """
    Description
    ----------
    This class is used to collate the dataset elements into batches for training.

    Inputs
    ----------

    """
    def __init__(
        self, 
        processor, 
        tokenizer,
        prompt, 
        max_sequence_length:int = 256
    ):
        self.processor = processor
        self.tokenizer = tokenizer
        self.prompt = prompt
        self.max_sequence_length = max_sequence_length
        self.tokenizer = get_tokenizer(processor)

    def __call__(self, features):
        images = [feature['image'] for feature in features]

        prompt_texts = [
            apply_chat_template(
                self.processor,
                build_messages(self.prompt),
                add_generation_prompt = True
            )
            for _ in features
        ]

        # deliberately skipping the full_texts part, as we want to make text from images

        batch = self.processor(
            images = images,
            padding = True,
            truncation = False,
            max_length = self.max_sequence_length,
            return_tensors = 'pt'
        )
        labels = batch['input_ids'].clone()



In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = get_tokenizer(processor)

collator = GraniteCardOCRCollator(
    processor = processor,
    tokenizer = tokenizer,
    max_sequence_length = MAX_SEQUENCE_LENGTH,
    prompt = CARD_EXTRACTION_PROMPT
)